# 03 Case Retrieval

Notebook ini digunakan untuk tahap ketiga sistem Case-Based Reasoning (CBR), yaitu membangun fungsi pencarian kasus lama yang paling mirip dengan kasus baru.

Input yang digunakan:
- File hasil tahap 2: `stage_02_case_representation_output.zip`
- Dataset utama: `data/processed/cases.csv`

Metode yang digunakan:
- TF-IDF Vectorization
- Cosine Similarity
- Train-test split 80:20 untuk menyiapkan query uji awal

Output utama:
- Fungsi `retrieve(query, k=5)`
- `data/eval/queries.json`
- `data/results/retrieval_results.csv`
- `data/processed/tfidf_vectorizer.pkl`
- `data/processed/tfidf_matrix.npz`
- `stage_03_case_retrieval_output.zip`


## 1. Instalasi Library

Cell ini memasang library yang dibutuhkan untuk membangun model retrieval berbasis TF-IDF dan cosine similarity.


In [ ]:
!pip -q install pandas numpy scikit-learn scipy joblib tqdm openpyxl

## 2. Import Library

Cell ini memanggil library yang digunakan untuk membaca data, memproses teks, membangun vektor TF-IDF, menghitung cosine similarity, dan menyimpan model retrieval.


In [ ]:
import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Library berhasil dimuat.")

## 3. Menyiapkan Struktur Folder

Cell ini menyiapkan struktur folder project. Folder `data/processed/` digunakan untuk membaca dataset kasus dan menyimpan model retrieval.


In [ ]:
BASE_DIR = Path("/content/cbr-putusan-penganiayaan")

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder project siap digunakan.")
print(BASE_DIR)

## 4. Upload dan Ekstrak Output Tahap 2

Cell ini meminta file `stage_02_case_representation_output.zip`. File tersebut berisi `cases.csv` yang akan digunakan sebagai dasar case retrieval.


In [ ]:
EXPECTED_ZIP_NAME = "stage_02_case_representation_output.zip"
zip_path = Path("/content") / EXPECTED_ZIP_NAME

if not zip_path.exists():
    try:
        from google.colab import files
        print("Silakan upload file stage_02_case_representation_output.zip")
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise FileNotFoundError("Tidak ada file yang diupload.")
        zip_path = Path("/content") / list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            "File ZIP tahap 2 belum tersedia. Upload file stage_02_case_representation_output.zip terlebih dahulu."
        ) from e

print(f"File ZIP digunakan: {zip_path}")

extract_temp = Path("/content/stage_02_extract_temp")
if extract_temp.exists():
    shutil.rmtree(extract_temp)
extract_temp.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_temp)

candidate_data_dirs = list(extract_temp.rglob("data"))

if len(candidate_data_dirs) == 0:
    raise FileNotFoundError("Folder data tidak ditemukan di dalam ZIP tahap 2.")

source_root = candidate_data_dirs[0].parent

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
shutil.copytree(source_root, BASE_DIR)

RAW_DIR = BASE_DIR / "data" / "raw"
TEXT_DIR = BASE_DIR / "data" / "text"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
EVAL_DIR = BASE_DIR / "data" / "eval"
RESULTS_DIR = BASE_DIR / "data" / "results"
LOG_DIR = BASE_DIR / "logs"

for folder in [RAW_DIR, TEXT_DIR, PROCESSED_DIR, EVAL_DIR, RESULTS_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

cases_csv_path = PROCESSED_DIR / "cases.csv"

if not cases_csv_path.exists():
    raise FileNotFoundError("File cases.csv tidak ditemukan. Pastikan tahap 2 sudah selesai.")

print("File cases.csv ditemukan:")
print(cases_csv_path)

## 5. Membaca Dataset Kasus

Cell ini membaca file `cases.csv` hasil tahap Case Representation. Data ini berisi metadata, ringkasan fakta, amar putusan, dan teks lengkap dari setiap kasus.


In [ ]:
cases_df = pd.read_csv(cases_csv_path)

print("Ukuran dataset:", cases_df.shape)
display(cases_df.head())

required_columns = ["case_id", "nomor_perkara", "ringkasan_fakta", "pasal", "amar_putusan"]

missing_cols = [col for col in required_columns if col not in cases_df.columns]
if missing_cols:
    raise ValueError(f"Kolom wajib belum tersedia: {missing_cols}")

if len(cases_df) != 30:
    print("PERINGATAN: Jumlah case bukan 30. Periksa kembali dataset.")

cases_df[["case_id", "nomor_perkara", "pasal"]].head(10)

## 6. Membuat Teks untuk Retrieval

Cell ini membuat kolom `retrieval_text` sebagai gabungan dari ringkasan fakta dan pasal. Kolom ini digunakan sebagai dasar pencarian kemiripan antar kasus.


In [ ]:
def safe_text(value):
    if pd.isna(value):
        return ""
    return str(value)

cases_df["retrieval_text"] = (
    cases_df["ringkasan_fakta"].apply(safe_text) + " " +
    cases_df["pasal"].apply(safe_text)
)

cases_df["retrieval_text_length"] = cases_df["retrieval_text"].apply(lambda x: len(str(x).split()))

cases_df[["case_id", "nomor_perkara", "retrieval_text_length", "retrieval_text"]].head()

## 7. Fungsi Preprocessing Teks

Cell ini membuat fungsi preprocessing untuk menormalkan teks sebelum diubah menjadi vektor TF-IDF. Proses ini meliputi lowercase, penghapusan karakter tidak penting, dan normalisasi spasi.


In [ ]:
def preprocess_text(text):
    """
    Membersihkan teks agar lebih stabil untuk proses TF-IDF.
    """
    if not isinstance(text, str):
        text = ""

    text = text.lower()

    # Normalisasi beberapa variasi kata penting
    text = text.replace("kuhpidana", "kuhp")
    text = text.replace("kuh pidana", "kuhp")
    text = text.replace("pidanaidana", "pidana")
    text = text.replace("nopember", "november")
    text = text.replace("pebruari", "februari")

    # Pertahankan huruf, angka, dan spasi
    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)

    # Normalisasi spasi
    text = re.sub(r"\\s+", " ", text).strip()

    return text


custom_stopwords = [
    "bahwa", "terdakwa", "saksi", "korban", "menimbang", "pengadilan",
    "negeri", "rantau", "prapat", "putusan", "nomor", "pid", "pn", "rap",
    "dengan", "dalam", "pada", "hari", "tanggal", "sekira", "wib",
    "atau", "setidak", "tidaknya", "tempat", "tersebut", "adalah",
    "sebagai", "berikut", "melakukan", "perbuatan", "sehingga",
    "akibat", "mengalami", "terhadap", "dari", "dan", "yang", "di",
    "ke", "untuk", "oleh", "karena", "ini", "itu", "lalu", "kemudian",
    "telah", "benar", "bahwasanya", "alias", "als"
]

cases_df["retrieval_text_clean"] = cases_df["retrieval_text"].apply(preprocess_text)

cases_df[["case_id", "retrieval_text_clean"]].head()


## 8. Membagi Data Train dan Test

Cell ini membagi data menjadi train dan test dengan rasio 80:20. Data test digunakan sebagai sumber query uji awal untuk menguji fungsi retrieval.


In [ ]:
train_df, test_df = train_test_split(
    cases_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Jumlah data train:", len(train_df))
print("Jumlah data test :", len(test_df))

display(test_df[["case_id", "nomor_perkara", "pasal"]].reset_index(drop=True))

## 9. Membangun Model TF-IDF

Cell ini membangun representasi vektor dokumen menggunakan TF-IDF. Setiap kasus akan diubah menjadi vektor numerik sehingga dapat dihitung kemiripannya menggunakan cosine similarity.


In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=False,
    preprocessor=None,
    tokenizer=None,
    stop_words=custom_stopwords,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(cases_df["retrieval_text_clean"])

print("Ukuran matriks TF-IDF:", tfidf_matrix.shape)
print("Jumlah fitur/kata:", len(vectorizer.get_feature_names_out()))

## 10. Menghitung Similarity Matrix

Cell ini menghitung cosine similarity antar seluruh kasus. Nilai similarity berada pada rentang 0 sampai 1. Semakin tinggi nilainya, semakin mirip dua kasus tersebut.


In [ ]:
similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=cases_df["case_id"],
    columns=cases_df["case_id"]
)

similarity_df.iloc[:5, :5]

## 11. Fungsi Retrieval

Cell ini mendefinisikan fungsi `retrieve(query, k=5)`. Fungsi ini menerima teks kasus baru, menghitung kemiripan dengan seluruh case base, lalu mengembalikan top-k kasus paling mirip.


In [ ]:
def retrieve(query: str, k: int = 5, exclude_case_id: str = None):
    """
    Mengambil top-k kasus paling mirip berdasarkan TF-IDF dan cosine similarity.

    Parameter:
    - query: teks kasus baru
    - k: jumlah kasus yang ingin diambil
    - exclude_case_id: case_id yang ingin dikeluarkan dari hasil, jika diperlukan

    Output:
    - DataFrame berisi top-k kasus paling mirip
    """
    clean_query = preprocess_text(query)
    query_vector = vectorizer.transform([clean_query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    result_df = cases_df.copy()
    result_df["similarity_score"] = similarities

    if exclude_case_id is not None:
        result_df = result_df[result_df["case_id"] != exclude_case_id]

    result_df = result_df.sort_values(by="similarity_score", ascending=False)

    selected_columns = [
        "case_id",
        "nomor_perkara",
        "tanggal_putusan",
        "terdakwa",
        "pasal",
        "amar_label",
        "similarity_score",
        "ringkasan_fakta",
        "amar_putusan"
    ]

    available_columns = [col for col in selected_columns if col in result_df.columns]

    return result_df[available_columns].head(k).reset_index(drop=True)


## 12. Uji Fungsi Retrieval dengan Query Manual

Cell ini menguji fungsi retrieval menggunakan contoh query manual. Hasil yang diharapkan adalah sistem menampilkan 5 putusan yang paling mirip dengan kasus baru.


In [ ]:
query_manual = '''
Terdakwa melakukan penganiayaan terhadap korban dengan cara memukul bagian wajah korban menggunakan tangan.
Akibat perbuatan tersebut, korban mengalami luka dan rasa sakit. Perbuatan terdakwa didakwa dengan Pasal 351 ayat (1) KUHP.
'''

manual_results = retrieve(query_manual, k=5)
manual_results[["case_id", "nomor_perkara", "pasal", "similarity_score"]]

## 13. Menampilkan Detail Top-1 Hasil Retrieval

Cell ini menampilkan detail kasus paling mirip dari query manual, termasuk ringkasan fakta dan amar putusannya.


In [ ]:
top_1 = manual_results.iloc[0]

print("CASE ID:", top_1["case_id"])
print("NOMOR PERKARA:", top_1["nomor_perkara"])
print("PASAL:", top_1["pasal"])
print("SIMILARITY SCORE:", round(top_1["similarity_score"], 4))

print("\nRINGKASAN FAKTA:")
print(str(top_1["ringkasan_fakta"])[:1500])

print("\nAMAR PUTUSAN:")
print(str(top_1["amar_putusan"])[:1500])

## 14. Membuat Query Uji Awal

Cell ini membuat 5–10 query uji dari data test. Query uji disimpan ke file `data/eval/queries.json` sesuai kebutuhan tahap retrieval.


In [ ]:
queries = []

for i, (_, row) in enumerate(test_df.reset_index(drop=True).iterrows(), start=1):
    query_text = str(row["ringkasan_fakta"])

    # Batasi panjang query agar tetap rapi
    if len(query_text) > 1600:
        query_text = query_text[:1600]

    queries.append({
        "query_id": f"query_{i:03d}",
        "source_case_id": row["case_id"],
        "ground_truth_case_id": row["case_id"],
        "ground_truth_nomor_perkara": row["nomor_perkara"],
        "query_text": query_text
    })

queries_path = EVAL_DIR / "queries.json"

with open(queries_path, "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)

print(f"Jumlah query uji dibuat: {len(queries)}")
print(f"File queries.json disimpan di: {queries_path}")

queries[:2]

## 15. Menjalankan Retrieval untuk Seluruh Query Uji

Cell ini menjalankan fungsi retrieval untuk semua query uji dan menyimpan hasil top-5 ke dalam file `retrieval_results.csv`.


In [ ]:
retrieval_rows = []

for query in tqdm(queries, desc="Menjalankan retrieval query uji"):
    results = retrieve(query["query_text"], k=5)

    top_case_ids = results["case_id"].tolist()
    top_nomor_perkara = results["nomor_perkara"].tolist()
    top_scores = results["similarity_score"].round(6).tolist()

    # Cek apakah ground truth muncul di top-k
    is_relevant_at_5 = query["ground_truth_case_id"] in top_case_ids
    rank = None

    if is_relevant_at_5:
        rank = top_case_ids.index(query["ground_truth_case_id"]) + 1

    retrieval_rows.append({
        "query_id": query["query_id"],
        "ground_truth_case_id": query["ground_truth_case_id"],
        "ground_truth_nomor_perkara": query["ground_truth_nomor_perkara"],
        "top_5_case_ids": ", ".join(top_case_ids),
        "top_5_nomor_perkara": ", ".join(top_nomor_perkara),
        "top_5_similarity_scores": ", ".join(map(str, top_scores)),
        "ground_truth_found_at_top_5": is_relevant_at_5,
        "ground_truth_rank": rank
    })

retrieval_results_df = pd.DataFrame(retrieval_rows)

retrieval_results_path = RESULTS_DIR / "retrieval_results.csv"
retrieval_results_df.to_csv(retrieval_results_path, index=False)

retrieval_results_df

## 16. Ringkasan Hasil Retrieval Awal

Cell ini menampilkan ringkasan awal hasil retrieval. Evaluasi lengkap menggunakan accuracy, precision, recall, dan F1-score akan dilakukan pada notebook tahap 5.


In [ ]:
top5_accuracy = retrieval_results_df["ground_truth_found_at_top_5"].mean()

print("Ringkasan retrieval awal:")
print("Jumlah query uji:", len(retrieval_results_df))
print("Ground truth ditemukan di top-5:", retrieval_results_df["ground_truth_found_at_top_5"].sum())
print("Top-5 accuracy awal:", round(top5_accuracy, 4))

rank_distribution = retrieval_results_df["ground_truth_rank"].value_counts(dropna=False).sort_index()
rank_distribution

## 17. Menyimpan Model Retrieval

Cell ini menyimpan vectorizer, matriks TF-IDF, dan index kasus. File ini akan dipakai kembali pada tahap Solution Reuse dan Evaluation.


In [ ]:
vectorizer_path = PROCESSED_DIR / "tfidf_vectorizer.pkl"
tfidf_matrix_path = PROCESSED_DIR / "tfidf_matrix.npz"
case_index_path = PROCESSED_DIR / "case_index.csv"
similarity_matrix_path = PROCESSED_DIR / "similarity_matrix.csv"

joblib.dump(vectorizer, vectorizer_path)
sparse.save_npz(tfidf_matrix_path, tfidf_matrix)

case_index_df = cases_df[["case_id", "nomor_perkara"]].copy()
case_index_df.to_csv(case_index_path, index=False)

similarity_df.to_csv(similarity_matrix_path)

print("Model retrieval berhasil disimpan:")
print("-", vectorizer_path)
print("-", tfidf_matrix_path)
print("-", case_index_path)
print("-", similarity_matrix_path)

## 18. Membuat Ringkasan Tahap 3

Cell ini membuat file ringkasan tahap 3 yang berisi jumlah kasus, jumlah query uji, metode retrieval, dan hasil top-5 accuracy awal.


In [ ]:
stage_03_summary = {
    "jumlah_case": len(cases_df),
    "jumlah_query_uji": len(queries),
    "metode_vectorization": "TF-IDF",
    "metode_similarity": "Cosine Similarity",
    "train_test_split": "80:20",
    "top5_accuracy_awal": top5_accuracy,
    "output_queries_json": str(queries_path),
    "output_retrieval_results": str(retrieval_results_path),
    "output_vectorizer": str(vectorizer_path),
    "output_tfidf_matrix": str(tfidf_matrix_path)
}

stage_03_summary_df = pd.DataFrame([stage_03_summary])
stage_03_summary_path = PROCESSED_DIR / "stage_03_summary.csv"
stage_03_summary_df.to_csv(stage_03_summary_path, index=False)

stage_03_summary_df

## 19. Membuat ZIP Output Tahap 3

Cell ini membuat file ZIP project setelah tahap 3 selesai. ZIP ini dapat digunakan untuk melanjutkan ke tahap 4, yaitu Case/Solution Reuse.


In [ ]:
output_zip = Path("/content/stage_03_case_retrieval_output.zip")

if output_zip.exists():
    output_zip.unlink()

shutil.make_archive(
    base_name=str(output_zip).replace(".zip", ""),
    format="zip",
    root_dir=BASE_DIR
)

print(f"ZIP output tahap 3 berhasil dibuat: {output_zip}")

## Kesimpulan Tahap 3

Pada tahap ini, sistem retrieval telah dibuat menggunakan TF-IDF dan cosine similarity. Fungsi `retrieve(query, k=5)` dapat menerima kasus baru dalam bentuk teks dan mengembalikan top-5 kasus yang paling mirip dari case base. Hasil tahap ini akan digunakan pada tahap berikutnya untuk mengambil solusi dari kasus lama berdasarkan amar putusan.
